In [170]:
import numpy as np
import sys
sys.path.append("../src")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import glob
import os
from utils import time_embedding_np, reward_simulate,simulate_data_raw,make_weighted_target,position_encoder
from utils import best_next_state,path_opt,preward_opt,compute_normalized_future_rewards
from treeple.experimental import StreamDecisionForest
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader,Subset, RandomSampler
import xgboost as xgb
from sklearn.model_selection import train_test_split

In [44]:
base_reward = 10.0
map_name = "short"
decay_rate = 0.6
reward_period = 10 
session_duration = 500000
rewards_in_period = []

total_steps = session_duration
for period_start in range(0, total_steps, reward_period):
    period_end = min(period_start + reward_period, total_steps)
    steps_in_period = np.arange(period_start, period_end)
    rewards_in_period.extend(base_reward * (decay_rate ** (steps_in_period - period_start)))

pattern = [1,1,1,1,1,1,1,2,3,4,5,5,5,5,5,5,5,4,3,2]
optimal_states = np.array(pattern*(session_duration // len(pattern))).reshape(-1,1)

In [45]:
def get_candidates(state, t):
    if state == 0:
        return [1, 0]
    elif state == 6:
        return [5, 6]
    else:
        # default 3-way split for illustration
        return [state-1, state, state+1]

def enumerate_paths(x0, t_inital = 0, t_prime=20):
    paths = [[x0]]
    for t in range(t_prime):
        new_paths = []
        for path in paths:
            curr = path[-1]
            for nxt in get_candidates(curr, t):
                new_paths.append(path + [nxt])
        paths = new_paths
    
    # Convert to DataFrame: each row is one path, columns t=0..T
    cols = [f"t={t_inital+i}" for i in range(t_prime+1)]
    df = pd.DataFrame(paths, columns=cols)
    return df


In [346]:
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden=(256, 128), p_dropout=0.2, use_sigmoid=True):
        super().__init__()
        layers = []
        d = input_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(p_dropout)]
            d = h
        layers += [nn.Linear(d, 1)]
        if use_sigmoid:           # keep preds in [0,1] since y is normalized
            layers += [nn.Sigmoid()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)        # (N, 1)

def mse_loss(pred, target,weight= None, eps=1e-12):
    # return torch.sqrt(nn.functional.mse_loss(pred, target) + eps)
    return nn.functional.mse_loss(pred, target)+ eps

# def mse_loss(pred, target, weight=None, eps=1e-12):
#     se = (pred.view(-1) - target.view(-1))**2
#     if weight is None:
#         return se.mean()
#     w = weight.view(-1).clamp_min(0)
#     return (w * se).sum() / (w.sum() + eps)

def train_regressor(model, train_loader, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_tr = float("inf")
    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        tot_mse, n_batches = 0.0, 0
        for batch in train_loader:
            xb, yb = batch
            wb = None
            
            xb = xb.to(device).float()
            yb = yb.to(device).float().view(-1, 1)  # ensure (N,1)

            opt.zero_grad()
            preds = model(xb)
            # print(preds[:5],yb[:5])
            loss = mse_loss(preds, yb, weight= None)
            loss.backward()
            opt.step()

            tot_mse += loss.item()
            # print(tot_mse)
            n_batches += 1
            # print(n_batches)
        # print(f"Epoch {ep:02d} | train MSE {tot_mse / max(1, n_batches):.6f}")
            
        # validation
        model.eval()
        val_rmse, m = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                if len(batch) == 3:
                    xb, yb, wb = batch
                    wb = wb.to(device).float().view(-1, 1)
                else:
                    xb, yb = batch
                    wb = None
                preds = model(xb)
                val_rmse += mse_loss(preds, yb, wb).item()
                m += 1
        val_rmse /= max(1, m)
        tr_mse = tot_mse / max(1, n_batches)
        # print(tot_mse)
        if ep % 100 == 0:
            print(f"Epoch {ep:02d} | train MSE {tr_mse:.6f} | val MSE {val_rmse:.6f}")
        
        if tot_mse <= best_tr:
            if val_rmse <= best_val:
                best_tr = tot_mse
                best_val = val_rmse
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                print(f"Best Mode: Epoch {ep:02d} | train MSE {tr_mse:.6f} | val MSE {best_val:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def train_regressor_stratify(model, train_loader_0,train_loader_1, val_loader, epochs=30, lr=1e-3, weight_decay=1e-4, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_tr = float("inf")
    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs+1):
        model.train()
        tot_mse, n_batches = 0.0, 0
        for batch0,batch1 in zip(train_loader_0,train_loader_1):
            # if len(batch0) == 3:
            #     xb, yb, wb = batch
            #     wb = wb.to(device).float().view(-1, 1)
            # else:
            # print(len(batch0))
            xb_0, yb_0 = batch0
            xb_1, yb_1 = batch1
            xb = torch.cat([xb_0, xb_1], dim=0)
            yb = torch.cat([yb_0, yb_1], dim=0)
            # print(xb.shape,yb.shape)
            wb = None
            
            xb = xb.to(device).float()
            yb = yb.to(device).float().view(-1, 1)  # ensure (N,1)

            opt.zero_grad()
            preds = model(xb)
            # print(preds[:5],yb[:5])
            loss = mse_loss(preds, yb, weight= None)
            loss.backward()
            opt.step()

            tot_mse += loss.item()
            # print(tot_mse)
            n_batches += 1
            # print(n_batches)
        # print(f"Epoch {ep:02d} | train MSE {tot_mse / max(1, n_batches):.6f}")
            
        # validation
        model.eval()
        val_rmse, m = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                if len(batch) == 3:
                    xb, yb, wb = batch
                    wb = wb.to(device).float().view(-1, 1)
                else:
                    xb, yb = batch
                    wb = None
                preds = model(xb)
                val_rmse += mse_loss(preds, yb, wb).item()
                m += 1
        val_rmse /= max(1, m)
        tr_mse = tot_mse / max(1, n_batches)
        # print(tot_mse)
        if ep % 100 == 0:
            print(f"Epoch {ep:02d} | train MSE {tr_mse:.6f} | val MSE {val_rmse:.6f}")
        
        if tot_mse <= best_tr:
            # if val_rmse <= best_val:
            best_tr = tot_mse
            best_val = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"Best Mode: Epoch {ep:02d} | train MSE {tr_mse:.6f} | val MSE {val_rmse:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def predict(model, X):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        if isinstance(X, torch.Tensor):
            X_t = X.to(device).float()
        else:
            X_t = torch.from_numpy(np.asarray(X, dtype=np.float32)).to(device)
        preds = model(X_t).cpu().squeeze(1)
    return preds

In [405]:
def train_model(
    X, Y, weight=None,
    model=None, 
    ratio_val=0.1,
    hidden=(512, 256),
    p_dropout=0.1,
    epochs=500,
    lr=1e-4,
    weight_decay=1e-4,
    batch_size_train=32,
    batch_size_val=None,     # None, using full val set
    use_sigmoid=False,
    seed = None
):
    n = X.shape[0]

    if weight is None:
        weight = np.ones(n, dtype=np.float32)
    elif np.isscalar(weight):
        weight = np.full(n, float(weight), dtype=np.float32)
    else:
        weight = np.asarray(weight, dtype=np.float32)

    
    # len_val = 1 # Ignoring Validating set
    len_val = max(1, int(n * ratio_val))
    idx_val = np.arange(n-len_val,n)
    idx_tr  = np.setdiff1d(np.arange(n), idx_val)
    # rng = np.random.default_rng(seed) if seed is not None else None
    # idx_val = (rng.choice(n, size=len_val, replace=False) 
    #            if rng is not None else np.random.choice(n, len_val, replace=False))
    # idx_tr  = np.setdiff1d(np.arange(n), idx_val)
    # idx_val = np.arange(n-len_val,n)
    # idx_tr  = np.setdiff1d(np.arange(n), idx_val)

    X_val, Y_val, w_val = X[idx_val], Y[idx_val], weight[idx_val]
    X_tr,  Y_tr, w_tr  = X[idx_tr],  Y[idx_tr], weight[idx_tr]

    train_ds = TensorDataset(
        torch.from_numpy(X_tr.astype(np.float32)),
        torch.from_numpy(Y_tr.astype(np.float32)),
        # torch.from_numpy(w_tr.astype(np.float32))
    )
    val_ds = TensorDataset(
        torch.from_numpy(X_val.astype(np.float32)),
        torch.from_numpy(Y_val.astype(np.float32)),
        # torch.from_numpy(w_val.astype(np.float32))
    )

    if batch_size_val is None:
        batch_size_val = len(val_ds)  # evaluate on all val at once

    train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True,  drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size_val,   shuffle=False, drop_last=False)
    print(hidden,hidden,hidden,epochs,lr,weight_decay)
    if model is None:
        print('initialize',X.shape[1])
        model = MLPRegressor(
            input_dim=X.shape[1],
            hidden=hidden,
            use_sigmoid=False,
            p_dropout = p_dropout
        )

    model = train_regressor(
        model, train_loader, val_loader,
        epochs=epochs, lr=lr, weight_decay=weight_decay
    )
    return model

def predict_paths(model, df_paths, t_start, t_prime):
    n_paths = df_paths.shape[0]
    preds = np.zeros((n_paths, t_prime+1))
    for i in range(t_prime+1):
        if df_paths.shape[1] == 1:
            cand_pe = position_encoder(df_paths, type="onehot")
        else:
            cand_pe = position_encoder(df_paths.iloc[:,i], type="onehot")
        curr_time = t_start + 1 + i
        time_emb = time_embedding_np(np.ones(cand_pe.shape[0]) * curr_time, tdim=50)
        features = np.hstack([cand_pe, time_emb])
        preds[:, i] = predict(model,features)
    return preds

def choose_next_state(model_action, model_value, current_state, t_now, t_prime_action, t_prime_value, gamma):
    """Pick best next state based on discounted rewards."""
    df_paths = enumerate_paths(x0=current_state, t_inital=t_now, t_prime=t_prime_action)
    df_paths_value = np.asarray(df_paths.iloc[:,-1]).reshape(-1,1)
    preds_action = predict_paths(model_action, df_paths, t_now, t_prime_action)
    preds_value = predict_paths(model_value, df_paths_value, t_now+t_prime_action ,t_prime_value)
    rewards_matrix = preds_action[:, 1:]
    rewards_matrix = np.hstack([rewards_matrix,preds_value])
    discounts = gamma ** np.arange(0, t_prime_action+1)
    total_disc = (rewards_matrix * discounts).sum(axis=1)
    
    best_idx = np.argmax(total_disc)
    return int(df_paths.iloc[best_idx, 1]), df_paths, rewards_matrix

def run_inference(model_action, model_value, start_state, T_p, t_prime_action, t_prime_value, gamma, rewards_in_period):
    """Generate future trajectory and compute regret."""
    current_state = start_state
    future_states = [current_state]

    for delta_t in range(100):
        next_state, df_paths, preds = choose_next_state(model_action, model_value, current_state, T_p + delta_t, t_prime_action, t_prime_value, gamma)
        future_states.append(next_state)
        current_state = next_state

    # compute rewards
    irewards_test = [reward_simulate(future_states[i], T_p+i, rewards_in_period) 
                     for i in range(1, len(future_states))]
    preward = compute_normalized_future_rewards(irewards_test, 100, gamma,normalization=True)

    # optimal path benchmark
    path, ireward_opt = path_opt(future_states[0], T_p, 100)
    preward_opt_test = compute_normalized_future_rewards(ireward_opt[1:], 100, gamma,normalization=True)

    pregret = (np.sum(preward_opt_test).item() - np.sum(preward).item()) / 100

    return pregret,preds

# Offline - Batch mode

In [ ]:
# Ts = [2**i for i in np.arange(7,10)]+[750]+[1024,1250,1500,2048]
Ts = [128]
# Ts = [1024,1250,1500]
gamma = 0.9
t_prime_action = 6
t_prime_value = 0
threshold = 6
batch_size_train = 32
epochs = 2000
normalizer = 1
PREGRETS = []

for rep in range(10):
    # one simulation per rep
    actions, states, irewards, times = simulate_data_raw(
        rewards_in_period=rewards_in_period,
        session_duration=3000,
        tdim=50, n_sessions=1, seed=515+rep
    )
    state_posencode_oh = position_encoder(states[:,0], type="onehot")
    new_state_oh = np.hstack([state_posencode_oh,states[:,1:]])
    np.random.shuffle(irewards)
    PREGRET = []
    for T in Ts:
        # initial dataset
        T_val = int(0.1*T)
        T_initial = T - T_val
        X_T = new_state_oh[:T,:]
        X_train = new_state_oh[:T_initial,:]
        Y_train_action = irewards[:T_initial]
        

        X_val = new_state_oh[T_initial:T,:]
        Y_val_action = np.array(irewards[T_initial:T]).reshape(-1,1)
        print(X_train.shape,Y_train_action.shape,X_val.shape,Y_val_action.shape)
        if T >= 1024:
            batch_size_train = 64
        batch_size_val = T_val
        train_ds = TensorDataset(torch.from_numpy(X_train.astype(np.float32)), 
                                 torch.from_numpy(Y_train_action.astype(np.float32)))
        val_ds   = TensorDataset(torch.from_numpy(X_val.astype(np.float32)), 
                                 torch.from_numpy(Y_val_action.astype(np.float32)))
        train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True, drop_last=False)
        val_loader   = DataLoader(val_ds,  batch_size=batch_size_val, shuffle=False, drop_last=False)
        mlp_action = MLPRegressor(input_dim=X_train.shape[1], hidden=(128,12), p_dropout=0.1, use_sigmoid=False)
        mlp_action = train_regressor(mlp_action, train_loader, val_loader, epochs=1500, lr=1e-2,weight_decay = 1e-4)

       
        # Y_train_value = make_weighted_target(Y_train_action, 0.9,T = T_initial,normalization = True)
        # Y_val_value = make_weighted_target(Y_val_action, 0.9,T = T_val,normalization = True)
        prewards = make_weighted_target(irewards[:T], gamma,T = T,normalization = False)
        prewards = prewards/normalizer
        is_pos = (prewards > threshold).astype(int)
        targets = is_pos
        train_idx, valid_idx= train_test_split(
        np.arange(len(targets)),
        test_size=0.1,
        shuffle=True,
        stratify=targets)

        X_train_value = X_T[train_idx,:]
        X_val_value = X_T[valid_idx,:]
        Y_train_value = prewards[train_idx]
        Y_val_value =  prewards[valid_idx]

        train_ds = TensorDataset(torch.from_numpy(X_train_value.astype(np.float32)), 
                                 torch.from_numpy(Y_train_value.astype(np.float32)))
        val_ds   = TensorDataset(torch.from_numpy(X_val_value.astype(np.float32)), 
                                 torch.from_numpy(Y_val_value.astype(np.float32)))
        train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True, drop_last=False)
        val_loader   = DataLoader(val_ds,  batch_size=batch_size_val, shuffle=False, drop_last=False)
        mlp_val = MLPRegressor(input_dim=X_train.shape[1], hidden=(512,256), p_dropout=0.1, use_sigmoid=False)
        mlp_val = train_regressor(mlp_val, train_loader, val_loader, epochs=2000, lr=1e-4,weight_decay = 1e-4)

        current_state = states[T,0]
        start_state = current_state
        pregret_val,pred = run_inference(mlp_action, mlp_val, start_state, T, t_prime_action, t_prime_value, gamma, rewards_in_period)
        print(f"rep {rep}, T_train {T}, pregret={pregret_val:.4f}")
        PREGRET.append(pregret_val)
    PREGRETS.append(PREGRET)

# Online 

In [165]:
# T = 2000
Ts = [2**i for i in np.arange(7,10)]+[750]+[1024,1250,1500,2048,4096]
gamma = 0.5
t_prime_action = 6
t_prime_value = 0
threshold = 1e-4
DELTA_T = 0 # If Zero, then it is offline, batch mode

epochs = 1000
PREGRETS = []

for rep in range(1):
    # one simulation per rep
    actions, states, irewards, times = simulate_data_raw(
        rewards_in_period=rewards_in_period,
        session_duration=2000,
        tdim=50, n_sessions=1, seed=515+rep
    )
    prewards = make_weighted_target(irewards, 0.5,T = T,normalization = False)
    for T in Ts:
        # initial dataset
        X_train = np.hstack([position_encoder(states[:T,0], type="onehot"), states[:T,1:]])
        Y_train_action = irewards[:T]
        mlp_action = train_model(X_train, Y_train_action,hidden = (128,128),lr = 1e-2,epochs = epochs)

        Y_train_value = prewards[:T]
        is_pos = (Y_train_value > threshold).astype(int)
        # pos = is_pos.sum()
        # neg = len(Y_train_value) - pos
        # w = np.where(Y_train_value > threshold, (neg / max(pos,1)), 1.0)
        print(is_pos)

        train_idx, valid_idx= train_test_split(
        np.arange(len(targets)),
        test_size=0.2,
        shuffle=True,
        stratify=targets)
        
        PREGRET = []
        current_state = states[T,0]
        for delta_t in range(1,DELTA_T+1):
            next_state, df_paths, preds = choose_next_state(mlp_action,mlp_val, current_state, T+delta_t, t_prime_action, t_prime_value, gamma)
            next_time = T+1+delta_t
            next_features = np.hstack([
                position_encoder(np.array([next_state]), type="onehot"),
                time_embedding_np(next_time, tdim=50).reshape(1,-1)
            ])
            next_reward = reward_simulate(next_state, next_time-1, rewards_in_period)

            ## PolicyIteration
            X_train = np.vstack([X_train, next_features])
            Y_train_action = np.vstack([Y_train_action.reshape(-1,1), [[next_reward]]])

            ## Offline
            # X_train = np.hstack([position_encoder(states[:T+delta_t,0], type="onehot"), states[:T+delta_t,1:]])
            # Y_train_action = irewards[:T+delta_t]

            mlp_action = train_model(X_train, Y_train_action,hidden = (128,128),lr = 1e-2,epochs = epochs)

            Y_train_value = make_weighted_target(Y_train_action, 0.5,T = T+delta_t,normalization = False)
            # is_pos = (Y_train_value > threshold).astype(int)
            # pos = is_pos.sum()
            # neg = len(Y_train_value) - pos
            # w = np.where(Y_train_value > threshold, (neg / max(pos,1)), 1.0)
            print(X_train.shape,Y_train_value.shape)
            mlp_val = train_model(X_train, Y_train_value,hidden = (512,256),lr = 3e-3,epochs = epochs)
            
            current_state = next_state  # move forward

        
            T_p = delta_t + T
            start_state = current_state
            pregret_val = run_inference(mlp_action, mlp_val, start_state, T_p,  t_prime_action, t_prime_value, gamma, rewards_in_period)
            print(f"rep {rep}, T_train {T_p}, pregret={pregret_val:.4f}")
        PREGRET.append(pregret_val)

    PREGRETS.append(PREGRET)


(128, 128) (128, 128) (128, 128) 1000 0.01 0.0001
initialize 57


/var/folders/7_/9pm8yyb11yl_smkwv7tk0ysc0000gn/T/ipykernel_56847/2660033491.py:19: UserWarning: Using a target size (torch.Size([12])) that is different to the input size (torch.Size([12, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return nn.functional.mse_loss(pred, target,weight)


Epoch 100 | train MSE 0.140111 | val MSE 8.505432
Epoch 200 | train MSE 0.073431 | val MSE 8.627974
Epoch 300 | train MSE 0.036486 | val MSE 8.446747
Epoch 400 | train MSE 0.173375 | val MSE 8.334069
Epoch 500 | train MSE 0.228218 | val MSE 8.276415
Epoch 600 | train MSE 0.023390 | val MSE 8.593121
Epoch 700 | train MSE 0.021993 | val MSE 8.520801
Epoch 800 | train MSE 0.105706 | val MSE 8.399473
Epoch 900 | train MSE 0.071020 | val MSE 8.698442
Epoch 1000 | train MSE 0.079808 | val MSE 8.676992
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


NameError: name 'targets' is not defined

In [64]:
T = 200
gamma = 0.5
t_prime_action = 6
t_prime_value = 0
threshold = 1e-4
DELTA_T = 400
PREGRETS_random = []

for rep in range(10):
    # one simulation per rep
    actions, states, irewards, times = simulate_data_raw(
        rewards_in_period=rewards_in_period,
        session_duration=2000,
        tdim=50, n_sessions=1, seed=515+rep
    )
    prewards = make_weighted_target(irewards, 0.5,T = T,normalization = False)

    # initial dataset
    X_train = np.hstack([position_encoder(states[:T,0], type="onehot"), states[:T,1:]])
    Y_train_action = irewards[:T]
    rf_action = train_model(X_train, Y_train_action, None)

    Y_train_value = prewards
    is_pos = (Y_train_value > threshold).astype(int)
    pos = is_pos.sum()
    neg = len(Y_train_value) - pos
    w = np.where(Y_train_value > threshold, (neg / max(pos,1)), 1.0)
    rf_value = train_model(X_train, Y_train_value, sample_weight=w)
    print(Y_train_value.shape)
    
    PREGRET = []
    current_state = states[T,0]
    for delta_t in range(1,DELTA_T+1):
        next_state, df_paths, preds = choose_next_state(rf_action,rf_value, current_state, T+delta_t, t_prime_action, t_prime_value, gamma)
        next_time = T+1+delta_t
        next_features = np.hstack([
            position_encoder(np.array([next_state]), type="onehot"),
            time_embedding_np(next_time, tdim=50).reshape(1,-1)
        ])
        next_reward = reward_simulate(next_state, next_time-1, rewards_in_period)

        ## PolicyIteration
        # X_train = np.vstack([X_train, next_features])
        # Y_train_action = np.vstack([Y_train_action.reshape(-1,1), [[next_reward]]])

        ## Offline
        X_train = np.hstack([position_encoder(states[:T+delta_t,0], type="onehot"), states[:T+delta_t,1:]])
        Y_train_action = irewards[:T+delta_t]

        rf_action = train_model(X_train, Y_train_action, rf_action)

        Y_train_value = make_weighted_target(Y_train_action, 0.5,T = T+delta_t,normalization = False)
        is_pos = (Y_train_value > threshold).astype(int)
        pos = is_pos.sum()
        neg = len(Y_train_value) - pos
        w = np.where(Y_train_value > threshold, (neg / max(pos,1)), 1.0)
        print(X_train.shape,Y_train_value.shape,w.shape)
        rf_value = train_model(X_train, Y_train_value, sample_weight=w)
        
        current_state = next_state  # move forward

    
        T_p = delta_t + T
        start_state = current_state
        pregret_val = run_inference(rf_action, rf_value, start_state, T_p,  t_prime_action, t_prime_value, gamma, rewards_in_period)
        print(f"rep {rep}, T_train {T_p}, pregret={pregret_val:.4f}")
        PREGRET.append(pregret_val)

    PREGRETS_random.append(PREGRET)


(200,)
(201, 57) (201,) (201,)
rep 0, T_train 201, pregret=0.6903
(202, 57) (202,) (202,)
rep 0, T_train 202, pregret=0.5714
(203, 57) (203,) (203,)
rep 0, T_train 203, pregret=0.9483
(204, 57) (204,) (204,)
rep 0, T_train 204, pregret=0.7091
(205, 57) (205,) (205,)
rep 0, T_train 205, pregret=0.7056
(206, 57) (206,) (206,)
rep 0, T_train 206, pregret=0.4750
(207, 57) (207,) (207,)
rep 0, T_train 207, pregret=0.4705
(208, 57) (208,) (208,)
rep 0, T_train 208, pregret=0.4716
(209, 57) (209,) (209,)
rep 0, T_train 209, pregret=0.2345
(210, 57) (210,) (210,)
rep 0, T_train 210, pregret=0.4925
(211, 57) (211,) (211,)
rep 0, T_train 211, pregret=0.3560
(212, 57) (212,) (212,)
rep 0, T_train 212, pregret=0.3440
(213, 57) (213,) (213,)
rep 0, T_train 213, pregret=0.4363
(214, 57) (214,) (214,)
rep 0, T_train 214, pregret=0.4340
(215, 57) (215,) (215,)
rep 0, T_train 215, pregret=0.6705
(216, 57) (216,) (216,)
rep 0, T_train 216, pregret=0.3352
(217, 57) (217,) (217,)
rep 0, T_train 217, pregr

In [417]:
PREGRETS_arr = np.vstack(PREGRETS)
# PREGRETS_random_arr = np.vstack(PREGRETS_random)
np.savez(
    f"../results/OnlinePAFM/PVAFM_NN_oh_w_time0_lookahead{6}_10reps.npz",
    pregrets = PREGRETS_arr,
    # pregrets_offline = PREGRETS_random_arr
    )
    

